# Лабораторная работа №2. Обработка данных парсинга и работа с pandas

**Дисциплина:** Новые технологии в РПС  
**Вариант:** №9 (Текстовые отзывы с сервиса *Otzovik*, классы оценок 1–5 звёзд)  
**Выполнил:** студент Кондратенко А.С.  

--- 
## Цель работы
Рефакторинг результатов сбора данных, создание файлов-аннотаций с помощью `pandas`, реорганизация файловой структуры датасета и проектирование собственных итераторов.

## 0. Подготовка окружения и импорт библиотек

In [ ]:
import os
from pathlib import Path
import random
import shutil
from typing import Dict, List, Optional, Tuple, Iterator
import pandas as pd

# При запуске в Google Colab автоматически клонируем датасет из репозитория
if not Path('dataset').exists():
    print('Инициализация рабочего каталога в среде Google Colab...')
    !git clone https://github.com/Alexeyphysics/ntrps_lab2.git _tmp_repo
    !cp -r _tmp_repo/dataset ./dataset
    !rm -rf _tmp_repo
    print('Датасет успешно скопирован в среду выполнения!')

print('Библиотеки импортированы, окружение готово.')

--- 
## 1. Задание 1. Формирование файла-аннотации исходного датасета
Создание CSV-файла с колонками: `absolute_path`, `relate_path`, `output_class`.

In [ ]:
def create_annotation(dataset_dir: str = 'dataset', output_csv: str = 'annotation.csv') -> pd.DataFrame:
    root = Path(dataset_dir).resolve()
    project_root = Path.cwd().resolve()
    records = []
    for class_folder in sorted(root.iterdir()):
        if class_folder.is_dir():
            cls_name = class_folder.name
            for f in sorted(class_folder.glob('*.txt')):
                records.append((
                    str(f.resolve()),
                    str(f.relative_to(project_root)),
                    cls_name
                ))
    cols = ['absolute_path', 'relate_path', 'output_class']
    df = pd.DataFrame(records, columns=cols)
    df.to_csv(output_csv, index=False, encoding='utf-8')
    return df

df_task1 = create_annotation()
print(f'Создана аннотация: {len(df_task1)} строк')
df_task1.head(10)

--- 
## 2. Задание 2. Копирование с именованием <class>_<index>.txt
Копирование файлов в единую папку `dataset_class/` и создание новой аннотации `annotation_class.csv`.

In [ ]:
def copy_class_prefixed(src_dir: str = 'dataset', dst_dir: str = 'dataset_class') -> pd.DataFrame:
    src = Path(src_dir).resolve()
    dst = Path(dst_dir).resolve()
    dst.mkdir(parents=True, exist_ok=True)
    project_root = Path.cwd().resolve()
    records = []
    for class_folder in sorted(src.iterdir()):
        if class_folder.is_dir():
            cls_name = class_folder.name
            for f in sorted(class_folder.glob('*.txt')):
                new_name = f'{cls_name}_{f.name}'
                target_f = dst / new_name
                shutil.copy2(f, target_f)
                records.append((
                    str(target_f.resolve()),
                    str(target_f.relative_to(project_root)),
                    cls_name
                ))
    cols = ['absolute_path', 'relate_path', 'output_class']
    df = pd.DataFrame(records, columns=cols)
    df.to_csv('annotation_class.csv', index=False, encoding='utf-8')
    return df

df_task2 = copy_class_prefixed()
print(f'Скопировано файлов: {len(df_task2)}')
df_task2.head(10)

--- 
## 3. Задание 3. Копирование со случайными номерами (0..10000)
Копирование без коллизий в папку `dataset_random/` и создание аннотации `annotation_random.csv`.

In [ ]:
def copy_random_numbers(src_dir: str = 'dataset', dst_dir: str = 'dataset_random') -> pd.DataFrame:
    src = Path(src_dir).resolve()
    dst = Path(dst_dir).resolve()
    dst.mkdir(parents=True, exist_ok=True)
    project_root = Path.cwd().resolve()
    all_files = []
    for class_folder in sorted(src.iterdir()):
        if class_folder.is_dir():
            for f in sorted(class_folder.glob('*.txt')):
                all_files.append((f, class_folder.name))
    # Генерация строго уникальных случайных чисел
    random_numbers = random.sample(range(0, 10001), len(all_files))
    records = []
    for (f, cls_name), num in zip(all_files, random_numbers):
        new_name = f'{str(num).zfill(5)}.txt'
        target_f = dst / new_name
        shutil.copy2(f, target_f)
        records.append((
            str(target_f.resolve()),
            str(target_f.relative_to(project_root)),
            cls_name
        ))
    cols = ['absolute_path', 'relate_path', 'output_class']
    df = pd.DataFrame(records, columns=cols)
    df.to_csv('annotation_random.csv', index=False, encoding='utf-8')
    return df

df_task3 = copy_random_numbers()
print(f'Скопировано файлов со случайными номерами: {len(df_task3)}')
df_task3.head(10)

--- 
## 4. Задание 4. Функция get_next_instance
Последовательное получение экземпляров заданного класса с возвратом `None` при исчерпании.

In [ ]:
class InstanceRegistry:
    def __init__(self, annotation_file: str = 'annotation.csv'):
        df = pd.read_csv(annotation_file, dtype={'output_class': str})
        self.data = {}
        self.pointers = {}
        for cls_name, grp in df.groupby('output_class'):
            self.data[str(cls_name)] = grp['absolute_path'].tolist()
            self.pointers[str(cls_name)] = 0

    def get_next(self, class_label: str) -> Optional[str]:
        cls_key = str(class_label)
        if cls_key not in self.data:
            return None
        idx = self.pointers[cls_key]
        items = self.data[cls_key]
        if idx >= len(items):
            return None
        path = items[idx]
        self.pointers[cls_key] = idx + 1
        return path

registry = InstanceRegistry('annotation.csv')
print('Получение всех экземпляров класса 1:')
count = 0
while True:
    instance = registry.get_next('1')
    if instance is None:
        print(f'Экземпляры исчерпаны. Всего получено: {count}. Следующий вызов: {registry.get_next("1")}')
        break
    count += 1
    print(f'[{count:02d}] {instance}')

--- 
## 5. Задание 5. Собственный класс-итератор ClassIterator
Реализация стандартного протокола итерации `__iter__` и `__next__` с выбросом исключения `StopIteration`.

In [ ]:
class ClassIterator(Iterator[str]):
    def __init__(self, class_label: str, annotation_file: str = 'annotation.csv'):
        self.class_label = str(class_label)
        df = pd.read_csv(annotation_file, dtype={'output_class': str})
        self.paths = df[df['output_class'] == self.class_label]['absolute_path'].tolist()
        self.cursor = 0

    def __iter__(self) -> 'ClassIterator':
        return self

    def __next__(self) -> str:
        if self.cursor >= len(self.paths):
            raise StopIteration
        path = self.paths[self.cursor]
        self.cursor += 1
        return path

# Проверка работы итератора для класса '5'
print('Итерация через цикл for по классу 5:')
iterator = ClassIterator('5')
for i, p in enumerate(iterator, 1):
    print(f'{i:02d} -> {p}')

# Проверка выброса StopIteration
try:
    next(iterator)
except StopIteration:
    print('Успех: StopIteration штатно перехвачен!')